# UAV High-Rate Localization & Wi-Fi FTM/RSSI Dataset Explorer

This notebook inspects the generated publication-grade dataset:
- **Master Clock:** 200 Hz uniform time grid (5 ms intervals)
- **High-rate Sensors:** Tri-axial Accelerometer & Gyroscope ($\Delta\mathbf{v}, \Delta\boldsymbol{\theta}$ strapdown pre-integration), 3-axis Magnetometer, Barometer
- **10 Hz GNSS:** Position and Velocity with Dilution of Precision
- **100 ms TDMA Wi-Fi Ranging:** 802.11mc/az FTM RTT, CFD Leading-Edge Multipath Bias, RSSI, SNR, Ricean K-factor across 5 Ground APs
- **Ground Truth & Estimator Benchmarks:** SO(3) SLERP Quaternions, Body Rates, Accelerometer/Gyroscope Bias Walks, and EKF2 Estimates

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

# Open Scenario 01
dataset_path = 'datasets/dataset_flight_scenario_01.h5'
f = h5py.File(dataset_path, 'r')

print('Dataset Metadata:')
for k, v in f['metadata'].attrs.items():
    print(f'  {k}: {v}')

time = f['time'][:]
pos_gt = f['ground_truth']['pos_ned'][:]
vel_gt = f['ground_truth']['vel_ned'][:]
ekf_pos = f['ekf2_estimates']['pos_ned'][:]
anchors = f['metadata']['anchor_positions_ned_m'][:]

print(f"Total Samples: {len(time)} (Duration: {time[-1]:.1f} s @ {f['metadata'].attrs['base_frequency_hz']} Hz)")

In [ ]:
# 1. Plot 3D Flight Trajectory vs 5 Terrestrial Anchors
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Down is negative Z in NED, plot height above ground as -Z
ax.plot(pos_gt[:, 0], pos_gt[:, 1], -pos_gt[:, 2], label='UAV Ground Truth Path', color='navy', lw=1.2)
ax.scatter(anchors[:, 0], anchors[:, 1], -anchors[:, 2], color='crimson', s=100, marker='^', label='Terrestrial APs (1-5)')

for i, ap in enumerate(anchors):
    ax.text(ap[0], ap[1], -ap[2] + 2.0, f'AP {i+1}', color='crimson', weight='bold')

ax.set_title('Scenario 01: 3D UAV Flight Path vs 5 Ground Access Points')
ax.set_xlabel('North [m]')
ax.set_ylabel('East [m]')
ax.set_zlabel('Height AGL [m]')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2. Plot Wi-Fi FTM Ranging Errors and Multipath Biases
wifi = f['wifi_ranging']
valid_mask = wifi['valid_flag'][:] == True

t_valid = time[valid_mask]
ap_id = wifi['active_ap_id'][valid_mask]
d_est = wifi['ftm_estimated_distance_m'][valid_mask]
d_true = wifi['ftm_true_distance_m'][valid_mask]
ranging_error = d_est - d_true
rssi_val = wifi['rssi_dbm'][valid_mask]

plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(t_valid[:400], ranging_error[:400], '.', color='teal', alpha=0.7)
plt.axhline(0, color='gray', linestyle='--')
plt.title('Wi-Fi FTM RTT Ranging Error ($d_{est} - d_{true}$) [First 400 Bursts]')
plt.ylabel('Error [m]')
plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
plt.plot(t_valid[:400], rssi_val[:400], '-', color='darkorange', lw=1.2)
plt.title('Received Signal Strength Indicator (RSSI)')
plt.xlabel('Time [s]')
plt.ylabel('RSSI [dBm]')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Plot High-Rate IMU Accelerometer and Strapdown Delta Velocity
imu = f['sensors_high_rate']
accel = imu['accel_xyz'][:]
delta_v = imu['delta_v_xyz'][:]

plt.figure(figsize=(12, 6))
plt.plot(time[:1000], accel[:1000, 0], label='Accel X (Front)', color='red', alpha=0.7)
plt.plot(time[:1000], accel[:1000, 1], label='Accel Y (Right)', color='green', alpha=0.7)
plt.plot(time[:1000], accel[:1000, 2], label='Accel Z (Down)', color='blue', alpha=0.7)
plt.title('High-Rate 200 Hz Calibrated Accelerometer Measurements [First 5 seconds]')
plt.xlabel('Time [s]')
plt.ylabel('Specific Force [m/s$^2$]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()